<a href="https://colab.research.google.com/github/Ravikumar0604/Generative_modeling_casestudy/blob/main/Homework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial homework 2: Imputation methods

In this homework, you will implement imputations to a dataset with missing values and measure the change of model predictions when compared with the original dataset

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt


### Load california housing dataset

In [ ]:

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [ ]:
y.head()

,MedHouseVal
0,4.526
1,3.585
2,3.521
3,3.413
4,3.422


### Predictions for dataset without missing values

In [ ]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=RANDOM_STATE)

RF = RandomForestRegressor(random_state=RANDOM_STATE)

RF.fit(X_train,y_train)

predictions_RF = RF.predict(X_test)

mse_RF = mean_squared_error(y_test, predictions_RF)

print("RF MSE = ", mse_RF)

LinReg = LinearRegression()

LinReg.fit(X_train,y_train)

predictions_LR = LinReg.predict(X_test)

mse_LR = mean_squared_error(y_test, predictions_LR)

print("LR MSE = ", mse_LR)

RF MSE =  0.2603403202420427
LR MSE =  0.5369686543372453


### Introducting missing values and re-training by removing rows with missing values

In [ ]:
# Load a random number generator
rng = np.random.default_rng(seed=42)
# Create a mask by selecting random numbers below some threshold
missing_mask = rng.random(X.shape) < 0.15  # 10% missing
X_missing = X.mask(missing_mask)
X_missing.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,NaN,2.555556,37.88,-122.23
1,NaN,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,NaN,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,NaN,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,NaN,2.181467,37.85,-122.25


In [ ]:
# Remove rows with NaNs
X_rem = X_missing.dropna()
y_rem = y[X_rem.index]

print("Number of rows in original dataset:", X.shape[0],"\n")
print("Number of rows after removing NaNs:", X_rem.shape[0],"\n")

Number of rows in original dataset: 20640 

Number of rows after removing NaNs: 5668 



In [ ]:
# Re-train with dataset where rows with missing values were removed
X_rem_train, X_rem_test, y_rem_train, y_rem_test = train_test_split(X_rem, y_rem, test_size=0.33, random_state=RANDOM_STATE)

RF_rem = RandomForestRegressor(random_state=RANDOM_STATE)

RF_rem.fit(X_rem_train,y_rem_train)

predictions_RF_rem = RF_rem.predict(X_rem_test)

mse_RF_rem = mean_squared_error(y_rem_test, predictions_RF_rem)

print("RF MSE after removal of missing rows = ", mse_RF_rem)

LinReg_rem = LinearRegression()

LinReg_rem.fit(X_rem_train,y_rem_train)

predictions_LR_rem = LinReg.predict(X_rem_test)

mse_LR_rem = mean_squared_error(y_rem_test, predictions_LR_rem)

print("LR MSE after removal of missing rows = ", mse_LR_rem)

RF MSE after removal of missing rows =  0.31343069103206517
LR MSE after removal of missing rows =  0.5341917883859029


## EXERCISE STARTS HERE

### The MICE imputation method

Use the MICE imputation method on the dataset with missing values and re-fit both the Random Forest and linear regression models. Calculate the difference in MSE between predictions with the original and imputed dataset for both models.

#### 1) Apply the default MICE method to the dataset with missing values. Compute the means and the standard deviations for each feature's distribution in both the original dataset (before adding random missing values) and the imputed dataset. Compute the difference between each mean and standard deviation

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Initialize and run the default MICE Imputer (IterativeImputer)
# Using RANDOM_STATE to keep the results deterministic
mice_imputer = IterativeImputer(random_state=RANDOM_STATE)
X_imputed_array = mice_imputer.fit_transform(X_missing)
X_imputed = pd.DataFrame(X_imputed_array, columns=X.columns, index=X.index)

# Calculate mean and standard deviation
stats_orig = pd.DataFrame({
    'mean_orig': X.mean(),
    'std_orig': X.std()
})

stats_imputed = pd.DataFrame({
    'mean_imputed': X_imputed.mean(),
    'std_imputed': X_imputed.std()
})

# Combine and compute differences
comparison_df = pd.concat([stats_orig, stats_imputed], axis=1)
comparison_df['mean_diff'] = comparison_df['mean_imputed'] - comparison_df['mean_orig']
# Difference in standard deviation
comparison_df['std_diff'] = comparison_df['std_imputed'] - comparison_df['std_orig']

display(comparison_df)

,mean_orig,std_orig,mean_imputed,std_imputed,mean_diff,std_diff
MedInc,3.870671,1.899822,3.875477,1.843598,0.004806,-0.056223
HouseAge,28.639486,12.585558,28.635307,11.825356,-0.004180,-0.760202
AveRooms,5.429000,2.474173,5.428637,2.505215,-0.000363,0.031042
AveBedrms,1.096675,0.473911,1.097047,0.470383,0.000372,-0.003528
Population,1425.476744,1132.462122,1426.505854,1058.655494,1.029110,-73.806628
AveOccup,3.070655,10.386050,3.061391,9.837383,-0.009264,-0.548666
Latitude,35.631861,2.135952,35.631707,2.110275,-0.000155,-0.025678
Longitude,-119.569704,2.003532,-119.566706,1.971866,0.002998,-0.031666


#### 2) Re-train the Random Forest and Linear Regression with the imputed data and compute the difference between MSEs in the imputed and original datasets for both models

In [ ]:
# Split the imputed data using the exact same random state and test size
X_imp_train, X_imp_test, y_imp_train, y_imp_test = train_test_split(
    X_imputed, y, test_size=0.33, random_state=RANDOM_STATE
)

# --- Random Forest Regressor on Imputed Data ---
RF_imp = RandomForestRegressor(random_state=RANDOM_STATE)
RF_imp.fit(X_imp_train, y_imp_train)
predictions_RF_imp = RF_imp.predict(X_imp_test)
mse_RF_imp = mean_squared_error(y_imp_test, predictions_RF_imp)

# --- Linear Regression on Imputed Data ---
LinReg_imp = LinearRegression()
LinReg_imp.fit(X_imp_train, y_imp_train)
predictions_LR_imp = LinReg_imp.predict(X_imp_test)
mse_LR_imp = mean_squared_error(y_imp_test, predictions_LR_imp)

# --- Calculate the differences ---
diff_RF_mse = mse_RF_imp - mse_RF
diff_LR_mse = mse_LR_imp - mse_LR

print(f"RF Baseline MSE: {mse_RF:.6f}")
print(f"RF Imputed MSE:  {mse_RF_imp:.6f}")
print(f"RF MSE Difference (Imputed - Baseline): {diff_RF_mse:+.6f}\n")

print(f"LR Baseline MSE: {mse_LR:.6f}")
print(f"LR Imputed MSE:  {mse_LR_imp:.6f}")
print(f"LR MSE Difference (Imputed - Baseline): {diff_LR_mse:+.6f}")

RF Baseline MSE: 0.260340
RF Imputed MSE:  0.409323
RF MSE Difference (Imputed - Baseline): +0.148982

LR Baseline MSE: 0.536969
LR Imputed MSE:  0.635947
LR MSE Difference (Imputed - Baseline): +0.098978


#### 3) Re-do the two steps above but using instead the MICE imputer with the DecisionTreeRegressor as estimator

Calculate the difference in MSE between the original models and the new models with the imputed dataset

ATTENTION: Make sure you set the random_state of the DecisionTreeRegressor to the same value as above to avoid variations with each run.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Initialize and run MICE Imputer with DecisionTreeRegressor as estimator
dt_estimator = DecisionTreeRegressor(random_state=RANDOM_STATE)
mice_imputer_dt = IterativeImputer(estimator=dt_estimator, random_state=RANDOM_STATE)

X_imputed_dt_array = mice_imputer_dt.fit_transform(X_missing)
X_imputed_dt = pd.DataFrame(X_imputed_dt_array, columns=X.columns, index=X.index)

# Split the DT-imputed data using the exact same random state and test size
X_dt_train, X_dt_test, y_dt_train, y_dt_test = train_test_split(
    X_imputed_dt, y, test_size=0.33, random_state=RANDOM_STATE
)

# --- Random Forest Regressor on DT-imputed Data ---
RF_dt = RandomForestRegressor(random_state=RANDOM_STATE)
RF_dt.fit(X_dt_train, y_dt_train)
predictions_RF_dt = RF_dt.predict(X_dt_test)
mse_RF_dt = mean_squared_error(y_dt_test, predictions_RF_dt)

# --- Linear Regression on DT-imputed Data ---
LinReg_dt = LinearRegression()
LinReg_dt.fit(X_dt_train, y_dt_train)
predictions_LR_dt = LinReg_dt.predict(X_dt_test)
mse_LR_dt = mean_squared_error(y_dt_test, predictions_LR_dt)

# --- Calculate differences against original baseline models ---
diff_RF_dt_mse = mse_RF_dt - mse_RF
diff_LR_dt_mse = mse_LR_dt - mse_LR

print(f"RF Baseline MSE: {mse_RF:.6f}")
print(f"RF DT-Imputed MSE: {mse_RF_dt:.6f}")
print(f"RF MSE Difference (DT-Imputed - Baseline): {diff_RF_dt_mse:+.6f}\n")

print(f"LR Baseline MSE: {mse_LR:.6f}")
print(f"LR DT-Imputed MSE: {mse_LR_dt:.6f}")
print(f"LR MSE Difference (DT-Imputed - Baseline): {diff_LR_dt_mse:+.6f}")

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


RF Baseline MSE: 0.260340
RF DT-Imputed MSE: 0.401847
RF MSE Difference (DT-Imputed - Baseline): +0.141507

LR Baseline MSE: 0.536969
LR DT-Imputed MSE: 0.624089
LR MSE Difference (DT-Imputed - Baseline): +0.087121


In [ ]:
abs_diff_rf = abs(mse_RF_imp - mse_RF)
print(f"Absolute difference for Random Forest MSE: {abs_diff_rf:.6f}")

Absolute difference for Random Forest MSE: 0.148982


In [ ]:
abs_diff_lr_dt = abs(mse_LR_dt - mse_LR)
print(f"Absolute difference for Linear Regression (DT-Imputed) MSE: {abs_diff_lr_dt:.6f}")

Absolute difference for Linear Regression (DT-Imputed) MSE: 0.087121
